# 03 · Transfer learning with ResNet18

ResNet18 pretrained on ImageNet (1.2M images, 1000 classes):

1. **Frozen backbone** — train only a new final layer (512 → 2).
2. **Full fine-tuning** — unfreeze all layers and train with a small learning rate.
3. Compare with the TinyVGG baseline trained from scratch.

Runtime: Colab, T4 GPU.

In [ ]:
import os
import random
from pathlib import Path
from timeit import default_timer as timer

import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
DATA_DIR = Path("data/cifake")
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"
N_TRAIN, N_TEST = 20_000, 4_000


def download_cifake_subset():
    """Download a balanced CIFAKE subset and save it in ImageFolder format:
    data/cifake/{train,test}/{FAKE,REAL}/*.png"""
    from datasets import load_dataset

    ds = load_dataset("dragonintelligence/CIFAKE-image-dataset")
    names = ds["train"].features["label"].names
    for split, n in [("train", N_TRAIN), ("test", N_TEST)]:
        part = ds[split].shuffle(seed=42).select(range(n))
        for i, ex in enumerate(part):
            cls = "FAKE" if "FAKE" in names[ex["label"]].upper() else "REAL"
            folder = DATA_DIR / split / cls
            folder.mkdir(parents=True, exist_ok=True)
            ex["image"].convert("RGB").save(folder / f"{i}.png")


if TRAIN_DIR.is_dir() and any(TRAIN_DIR.rglob("*.png")):
    print("Data already downloaded.")
else:
    download_cifake_subset()
    print("Done.")

In [ ]:
def accuracy_fn(y_true, y_pred):
    return torch.eq(y_true, y_pred).sum().item() / len(y_pred) * 100


def train_step(model, dataloader, loss_fn, optimizer, accuracy_fn, device):
    model.train()
    train_loss, train_acc = 0, 0
    for X, y in dataloader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        loss = loss_fn(logits, y)
        train_loss += loss.item()
        train_acc += accuracy_fn(y, logits.argmax(dim=1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    train_loss /= len(dataloader)
    train_acc /= len(dataloader)
    return train_loss, train_acc


def test_step(model, dataloader, loss_fn, accuracy_fn, device):
    model.eval()
    test_loss, test_acc = 0, 0
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            test_loss += loss_fn(logits, y).item()
            test_acc += accuracy_fn(y, logits.argmax(dim=1))
    test_loss /= len(dataloader)
    test_acc /= len(dataloader)
    return test_loss, test_acc


def train(model, train_dataloader, test_dataloader, loss_fn, optimizer, epochs, device):
    results = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
    for epoch in range(epochs):
        train_loss, train_acc = train_step(model, train_dataloader, loss_fn, optimizer, accuracy_fn, device)
        test_loss, test_acc = test_step(model, test_dataloader, loss_fn, accuracy_fn, device)
        print(f"epoch {epoch}: train loss {train_loss:.4f}, acc {train_acc:.1f}% | "
              f"test loss {test_loss:.4f}, acc {test_acc:.1f}%")
        results["train_loss"].append(train_loss)
        results["train_acc"].append(train_acc)
        results["test_loss"].append(test_loss)
        results["test_acc"].append(test_acc)
    return results


def n_params(model, trainable_only=False):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


loss_fn = nn.CrossEntropyLoss()

## Pretrained weights and their transforms

Images must be prepared exactly as during ImageNet training: resize to 256, center crop 224, ImageNet normalization.

In [ ]:
weights = torchvision.models.ResNet18_Weights.DEFAULT
auto_transforms = weights.transforms()
print(auto_transforms)

train_data = datasets.ImageFolder(root=TRAIN_DIR, transform=auto_transforms)
test_data = datasets.ImageFolder(root=TEST_DIR, transform=auto_transforms)
class_names = train_data.classes

train_dataloader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_dataloader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

## 1. Frozen backbone

In [ ]:
model_rn = torchvision.models.resnet18(weights=weights).to(device)
print(model_rn.fc)    # Linear(512 -> 1000): one output per ImageNet class

torch.manual_seed(42)
for param in model_rn.parameters():
    param.requires_grad = False

model_rn.fc = nn.Linear(in_features=512, out_features=len(class_names)).to(device)

print("total params:", n_params(model_rn), "| trainable:", n_params(model_rn, trainable_only=True))

In [ ]:
optimizer = torch.optim.Adam(params=model_rn.parameters(), lr=0.001)

start = timer()
rn_frozen_results = train(model_rn, train_dataloader, test_dataloader, loss_fn, optimizer, epochs=3, device=device)
rn_frozen_time = timer() - start

## 2. Full fine-tuning

In [ ]:
for param in model_rn.parameters():
    param.requires_grad = True

optimizer_ft = torch.optim.Adam(params=model_rn.parameters(), lr=0.0001)

start = timer()
rn_ft_results = train(model_rn, train_dataloader, test_dataloader, loss_fn, optimizer_ft, epochs=2, device=device)
rn_ft_time = timer() - start

In [ ]:
MODEL_PATH = Path("models") / "resnet18_cifake.pth"
MODEL_PATH.parent.mkdir(exist_ok=True)
torch.save(model_rn.state_dict(), MODEL_PATH)

## 3. Baseline: TinyVGG from scratch (same session, same GPU)

TinyVGG uses its own 32×32 data loaders.

In [ ]:
class TinyVGG(nn.Module):
    """Small CNN in the style of VGG: 2 conv blocks + linear classifier.
    Input: [batch, 3, 32, 32] -> output: [batch, output_shape] logits."""

    def __init__(self, input_shape, hidden_units, output_shape):
        super().__init__()
        self.block_1 = nn.Sequential(
            nn.Conv2d(input_shape, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                      # 32 -> 16
        )
        self.block_2 = nn.Sequential(
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_units, hidden_units, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),                      # 16 -> 8
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(hidden_units * 8 * 8, output_shape),
        )

    def forward(self, x):
        return self.classifier(self.block_2(self.block_1(x)))

In [ ]:
tv_train = DataLoader(
    datasets.ImageFolder(TRAIN_DIR, transform=transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5), transforms.ToTensor()])),
    batch_size=64, shuffle=True)
tv_test = DataLoader(datasets.ImageFolder(TEST_DIR, transform=transforms.ToTensor()), batch_size=64)

torch.manual_seed(42)
model_tv = TinyVGG(input_shape=3, hidden_units=10, output_shape=len(class_names)).to(device)

start = timer()
tv_results = train(model_tv, tv_train, tv_test, loss_fn,
                   torch.optim.Adam(model_tv.parameters(), lr=0.001), epochs=5, device=device)
tv_time = timer() - start

## Comparison

In [ ]:
import pandas as pd

pd.DataFrame({
    "model": ["TinyVGG from scratch", "ResNet18, frozen backbone", "ResNet18, full fine-tuning"],
    "trainable params": [n_params(model_tv), 512 * len(class_names) + len(class_names), n_params(model_rn)],
    "epochs": ["5", "3", "3 + 2"],
    "test acc, %": [round(tv_results["test_acc"][-1], 1),
                    round(rn_frozen_results["test_acc"][-1], 1),
                    round(rn_ft_results["test_acc"][-1], 1)],
    "train time, s": [round(tv_time), round(rn_frozen_time), round(rn_frozen_time + rn_ft_time)],
})

## Findings

- A **frozen ImageNet backbone** performs about the same as a tiny CNN trained from scratch: ImageNet features describe *what* is in the image, while AI-generated images differ in low-level pixel artifacts (which are also blurred by upscaling 32×32 → 224×224).
- **Full fine-tuning** lets the network adapt its features to the real-vs-generated task and gives the best accuracy.
- Trade-off: the fine-tuned ResNet18 has ~2,600× more parameters than TinyVGG and is slower at inference.